# MODIS Land Surface Temperature: Search, Download, and Regional Analysis

This tutorial demonstrates a complete workflow using `earthaccess` to retrieve **MODIS MOD11A2** (8-day Land Surface Temperature & Emissivity, 1 km) data and perform a regional analysis with `rasterio`, `geopandas`, and `rasterstats`.

**What you will learn:**
- Authenticate with NASA Earthdata using `earthaccess`
- Search and download MODIS MOD11A2 HDF tiles by bounding box and date range
- Mosaic multi-tile data into a single raster
- Convert Kelvin to Celsius and reproject to WGS84
- Clip the raster to a regional boundary shapefile
- Compute district-level zonal statistics with `rasterstats`
- Visualise the results with `geopandas` and `matplotlib`

**Dataset:** [MOD11A2 v061](https://doi.org/10.5067/MODIS/MOD11A2.061) — MODIS/Terra Land Surface Temperature/Emissivity 8-Day L3 Global 1 km SIN Grid

**Prerequisites:**
- A free [NASA Earthdata account](https://urs.earthdata.nasa.gov/)
- The packages listed in the cell below
- A boundary shapefile for your region of interest (this example uses [GADM Pakistan Level 3](https://gadm.org/))

## Install dependencies

In [ ]:
%pip install earthaccess numpy geopandas matplotlib rasterio rasterstats GDAL affine pyogrio --quiet

## 1. Import libraries

In [ ]:
import os
import re
import glob
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from osgeo import gdal
from rasterstats import zonal_stats

import earthaccess

print(f"earthaccess version: {earthaccess.__version__}")

## 2. Authenticate with NASA Earthdata

`earthaccess.login()` looks for credentials in the following order:
1. `~/.netrc` file
2. `EARTHDATA_USERNAME` / `EARTHDATA_PASSWORD` environment variables
3. Interactive prompt (if `strategy='interactive'`)

Register for a free account at [urs.earthdata.nasa.gov](https://urs.earthdata.nasa.gov/) if you do not have one.

In [ ]:
auth = earthaccess.login()
print("Authentication successful:", auth.authenticated)

## 3. Configure the search parameters

We search for **MOD11A2 v061** granules over Pakistan for April 2025.

The bounding box covers Pakistan: `(min_lon, min_lat, max_lon, max_lat)` = `(60, 23, 78, 38)`.

> **Tip:** You can find the MODIS sinusoidal tile IDs for any region using the [MODIS Tile Calculator](https://modis-land.appdata.nasa.gov/MODLAND_grid.html). Pakistan spans tiles h22–h25, v05–v06.

In [ ]:
# Output directories
base_dir = "modis_lst_output"
download_dir = os.path.join(base_dir, "hdf")
output_dir = os.path.join(base_dir, "geotiff")
os.makedirs(download_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# Search parameters
SHORT_NAME = "MOD11A2"
VERSION = "061"
START_DATE = "2025-04-01"
END_DATE = "2025-04-30"
BOUNDING_BOX = (60, 23, 78, 38)  # Pakistan: (W, S, E, N)

## 4. Search and download MODIS granules

In [ ]:
results = earthaccess.search_data(
    short_name=SHORT_NAME,
    version=VERSION,
    cloud_hosted=True,
    temporal=(START_DATE, END_DATE),
    bounding_box=BOUNDING_BOX,
)
print(f"Found {len(results)} granules")

In [ ]:
files = earthaccess.download(results, download_dir)
print(f"Downloaded {len(files)} files to: {download_dir}")

## 5. Mosaic tiles into a single raster

MODIS data is distributed as sinusoidal grid tiles. We group files by date, position each tile in a mosaic array, extract the `LST_Day_1km` subdataset, apply the scale factor (`× 0.02`), and convert from Kelvin to Celsius (`− 273.15`).

In [ ]:
# Tile layout — adjust if your bounding box spans different tiles
TILE_POSITIONS = {
    "h22v05": (0, 0), "h23v05": (0, 1), "h24v05": (0, 2), "h25v05": (0, 3),
    "h22v06": (1, 0), "h23v06": (1, 1), "h24v06": (1, 2), "h25v06": (1, 3),
}
TILE_SIZE = 1200
mosaic_rows = max(p[0] for p in TILE_POSITIONS.values()) + 1
mosaic_cols = max(p[1] for p in TILE_POSITIONS.values()) + 1
mosaic_shape = (TILE_SIZE * mosaic_rows, TILE_SIZE * mosaic_cols)

# Group HDF files by acquisition date
hdf_files = sorted(glob.glob(os.path.join(download_dir, "*.hdf")))
date_tile_dict = defaultdict(list)
pattern = re.compile(r"MOD11A2\.A(\d{7})\.(h\d{2}v\d{2})\.\d{3}\.\d{13}\.hdf")

for f in hdf_files:
    m = pattern.search(os.path.basename(f))
    if m:
        date_tile_dict[m.group(1)].append((m.group(2), f))

print(f"Found {len(hdf_files)} HDF files across {len(date_tile_dict)} dates")

In [ ]:
mosaic_stack = []
valid_dates = []
first_gt = first_proj = None

for date_str, tile_files in date_tile_dict.items():
    mosaic = np.full(mosaic_shape, np.nan, dtype=np.float32)
    n_tiles = 0

    for tile, hdf_path in tile_files:
        if tile not in TILE_POSITIONS:
            continue
        hdf = gdal.Open(hdf_path)
        lst_path = next(
            (s[0] for s in hdf.GetSubDatasets() if "LST_Day_1km" in s[0]), None
        )
        if lst_path is None:
            continue
        ds = gdal.Open(lst_path)
        arr = ds.ReadAsArray().astype(np.float32)
        # Scale factor 0.02, convert K → °C; fill value 0 → NaN
        arr = np.where(arr == 0, np.nan, arr * 0.02 - 273.15)
        if first_gt is None:
            first_gt = ds.GetGeoTransform()
            first_proj = ds.GetProjection()
        row, col = TILE_POSITIONS[tile]
        mosaic[
            row * TILE_SIZE : row * TILE_SIZE + TILE_SIZE,
            col * TILE_SIZE : col * TILE_SIZE + TILE_SIZE,
        ] = arr
        n_tiles += 1

    if n_tiles > 0:
        mosaic_stack.append(mosaic)
        valid_dates.append(date_str)
        print(f"  {date_str}: mosaicked {n_tiles} tiles")

print(f"\nTotal composites: {len(mosaic_stack)}")

## 6. Compute mean LST and save as GeoTIFF

In [ ]:
stacked = np.stack(mosaic_stack)
mean_lst = np.nanmean(stacked, axis=0)

print(f"Mean LST range: {np.nanmin(mean_lst):.1f}°C to {np.nanmax(mean_lst):.1f}°C")
print(f"Date range: {valid_dates[0]} → {valid_dates[-1]}")

# Save as unprojected GeoTIFF
temp_unproj = os.path.join(output_dir, "mean_lst_sinusoidal.tif")
driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create(temp_unproj, mosaic_shape[1], mosaic_shape[0], 1, gdal.GDT_Float32)
out_ds.SetGeoTransform(first_gt)
out_ds.SetProjection(first_proj)
out_ds.GetRasterBand(1).WriteArray(mean_lst)
out_ds.GetRasterBand(1).SetNoDataValue(float("nan"))
out_ds.FlushCache()
del out_ds

# Reproject to WGS84 (EPSG:4326)
temp_wgs84 = os.path.join(output_dir, "mean_lst_wgs84.tif")
gdal.Warp(temp_wgs84, temp_unproj, dstSRS="EPSG:4326")
print("Reprojected raster saved to:", temp_wgs84)

## 7. Clip raster to a regional boundary

Load your boundary shapefile and clip the raster to the region of interest. Here we use the GADM Pakistan Level-3 boundary (district level). Download it from [gadm.org](https://gadm.org/download_country.html) and place it alongside this notebook.

In [ ]:
# Load boundary — replace the path with your own shapefile
SHAPEFILE = "gadm41_PAK_3.shp"  # GADM Pakistan Level-3 districts
gdf = gpd.read_file(SHAPEFILE).to_crs("EPSG:4326")
region_geom = [gdf.union_all().__geo_interface__]

with rasterio.open(temp_wgs84) as src:
    clipped, clipped_transform = mask(src, region_geom, crop=True)
    meta = src.meta.copy()
    meta.update(
        height=clipped.shape[1],
        width=clipped.shape[2],
        transform=clipped_transform,
    )

clipped_path = os.path.join(output_dir, "mean_lst_clipped.tif")
with rasterio.open(clipped_path, "w", **meta) as dst:
    dst.write(clipped)

print("Clipped raster saved to:", clipped_path)

## 8. Compute district-level zonal statistics

`rasterstats.zonal_stats` calculates summary statistics (mean, min, max, etc.) for each polygon in the boundary GeoDataFrame against the clipped raster.

In [ ]:
stats = zonal_stats(gdf, clipped_path, stats=["mean", "min", "max"], geojson_out=False)
gdf["mean_LST_C"] = [s["mean"] for s in stats]
gdf["min_LST_C"] = [s["min"] for s in stats]
gdf["max_LST_C"] = [s["max"] for s in stats]

print("Top 5 hottest districts:")
print(gdf[["NAME_3", "mean_LST_C"]].nlargest(5, "mean_LST_C").to_string(index=False))
print("\nTop 5 coldest districts:")
print(gdf[["NAME_3", "mean_LST_C"]].nsmallest(5, "mean_LST_C").to_string(index=False))

## 9. Visualise the results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Map: district-level mean LST
gdf.plot(
    column="mean_LST_C",
    cmap="RdYlBu_r",
    legend=True,
    edgecolor="black",
    linewidth=0.3,
    ax=axes[0],
    legend_kwds={"label": "Mean LST (°C)"},
)
axes[0].set_title(f"Mean Land Surface Temperature — {START_DATE[:7]}", fontsize=13)
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].axis("on")

# Bar chart: top 10 hottest vs coldest
top10_hot = gdf.nlargest(10, "mean_LST_C")
top10_cold = gdf.nsmallest(10, "mean_LST_C")

axes[1].barh(top10_hot["NAME_3"], top10_hot["mean_LST_C"], color="#d62728", label="Hottest")
axes[1].barh(top10_cold["NAME_3"], top10_cold["mean_LST_C"], color="#1f77b4", label="Coldest")
axes[1].set_xlabel("Mean LST (°C)")
axes[1].set_title("Top 10 Hottest & Coldest Districts", fontsize=13)
axes[1].legend()
axes[1].axvline(0, color="black", linewidth=0.8, linestyle="--")

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "modis_lst_results.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

## 10. Save vector output

In [ ]:
output_gpkg = os.path.join(output_dir, "district_lst_stats.gpkg")
gdf[["NAME_1", "NAME_2", "NAME_3", "mean_LST_C", "min_LST_C", "max_LST_C", "geometry"]].to_file(
    output_gpkg, driver="GPKG"
)
print("Vector output saved to:", output_gpkg)

## Summary

In this tutorial we:

1. Used `earthaccess` to **search and download** 24 MODIS MOD11A2 HDF granules covering Pakistan in April 2025
2. **Mosaicked** multi-tile data and computed a mean LST composite, converting from raw digital numbers (Kelvin × 0.02) to degrees Celsius
3. **Reprojected** the result to WGS84 and **clipped** it to the Pakistan boundary
4. Computed **district-level zonal statistics** with `rasterstats` — identifying Kech (Balochistan) as the hottest district (~45.7 °C) and high-elevation Gilgit-Baltistan districts as the coldest
5. Saved both raster (GeoTIFF) and vector (GeoPackage) outputs

The same workflow can be adapted to any region by changing `BOUNDING_BOX`, `START_DATE`/`END_DATE`, and the boundary shapefile. Other MODIS products (e.g., MOD13A2 for NDVI, MOD09GA for surface reflectance) can be used by changing `SHORT_NAME`.

### References

- Wan, Z., Hook, S., Hulley, G. (2021). *MODIS/Terra Land Surface Temperature/Emissivity 8-Day L3 Global 1 km SIN Grid V061*. NASA EOSDIS LP DAAC. <https://doi.org/10.5067/MODIS/MOD11A2.061>
- GADM (2023). *Database of Global Administrative Areas*. <https://gadm.org>
- earthaccess documentation: <https://earthaccess.readthedocs.io>